# Validating short implicit-solvent MD against SAXS and NMR — bovine pancreatic ribonuclease A

Reference system: **bovine pancreatic ribonuclease A**, UniProt `P61823`.

This notebook runs the full loop end to end:

1. survey the candidate SASBDB entries and **check they describe the same molecule**
2. load one experimental scattering curve, normalised to 1/A
3. **check the curve against itself** — recompute Guinier Rg from the raw data,
   and look at the Kratky plot before assuming the particle is folded
4. source atomic coordinates through UniProt
5. prepare and simulate in implicit solvent
6. compute an ensemble SAXS profile and fit it
7. compare the distance distribution P(r) with the deposited GNOM transform
8. compare NMR chemical shifts to the simulated secondary structure, against
   two controls
9. run a point mutant **with replicates**, to see whether the method can
   resolve the change at all

RNase A is the closest available analogue to lysozyme: a small, rigid, exhaustively studied enzyme with a sub-Angstrom structure. It exists here to answer whether the lysozyme result is a property of lysozyme or of the method. A conclusion that holds for one protein and not the other is a different kind of finding from one that holds for both.

**Expect roughly 27–37 minutes end to end on a laptop CPU**,
almost all of it in section 7, which runs 6 short simulations to get
replicates. Sections 1–6 take a few minutes. Set `N_REPLICAS = 1` there if you
only want to see the pipeline work — but read what that section says about why
one run per group is not a control before trusting any number it then prints.

> **Use a kernel from the `saxsnmr` environment.**
>
> On Windows, conda keeps its compiled libraries in `<env>\Library\bin`, which only
> joins the search path when the environment is activated. A Jupyter kernel launches
> `python.exe` directly, so it normally misses them, and the first call into NumPy's
> linear algebra then kills the kernel with no exception and no traceback — reported
> only as *"the kernel crashed"*.
>
> Importing `saxsnmr` repairs PATH itself, so the notebook works whichever way the
> kernel was started. That is why the cell below imports it before anything else.

In [ ]:
# Environment check. Run this first.
#
# Importing saxsnmr before anything else matters: on Windows it repairs PATH
# so the conda environment's compiled libraries can be found. Without that,
# the first call into NumPy's linear algebra kills the kernel outright -- no
# exception, no traceback, just "the kernel crashed".
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import saxsnmr

# A kernel that imported saxsnmr before the package was last edited keeps the
# old module object, and new names are simply absent. That surfaces much
# later as an opaque ImportError in whichever cell first needs one, so check
# the API surface here instead.
REQUIRED = ["Fetcher", "prepare", "run_md", "ensemble_fit", "guinier",
            "pair_distribution", "threshold_scan", "shift_dispersion",
            "sans_tables", "sans_profile", "match_point_d2o"]
missing = [n for n in REQUIRED if not hasattr(saxsnmr, n)]
if missing:
    raise ImportError(
        f"the imported saxsnmr is out of date -- missing {missing}.\n"
        "The kernel cached it before the package was edited. Either restart\n"
        "the kernel, or run this to reload without losing your variables:\n\n"
        "    import sys\n"
        "    for m in [k for k in sys.modules\n"
        "              if k == 'saxsnmr' or k.startswith('saxsnmr.')]:\n"
        "        del sys.modules[m]\n\n"
        "then re-run this cell. Reloading only the top-level package is not\n"
        "enough: the submodules stay cached and you get a mix of versions.")

print("python:", sys.executable)
print("saxsnmr API complete:", len(REQUIRED), "entry points present")
if saxsnmr.DLL_PATH_ADDED:
    print("repaired PATH with", len(saxsnmr.DLL_PATH_ADDED), "library directories")
else:
    print("PATH already complete (environment was activated)")

checks = {
    "numpy linalg":   "import numpy as np; np.linalg.lstsq(np.eye(3), np.ones(3), rcond=None)",
    "scipy optimize": "from scipy.optimize import least_squares; least_squares(lambda p: p - 3.0, [0.0])",
    # nnls is a separate compiled entry point used by the SANS fit. Checking
    # least_squares does not exercise it, and anything that aborts the
    # process here would otherwise do so mid-notebook with no traceback.
    "scipy nnls":     ("import numpy as np; from scipy.optimize import nnls; "
                       "nnls(np.eye(3), np.ones(3))"),
    "mdtraj":         "import mdtraj",
    "openmm import":  "import openmm",
    "openmm runtime": ("import openmm as mm; s = mm.System(); s.addParticle(1.0); "
                       "mm.Context(s, mm.VerletIntegrator(0.001), "
                       "mm.Platform.getPlatformByName('CPU'))"),
    "pdbfixer":       "import pdbfixer",
}

failed = []
for name, snippet in checks.items():
    try:
        exec(snippet, {})
        print(f"  OK    {name}")
    except Exception as exc:
        print(f"  FAIL  {name}: {type(exc).__name__}: {exc}")
        failed.append(name)

if failed:
    print()
    print("Unusable:", ", ".join(failed))
    print("Select the 'Python (saxsnmr)' kernel, listed under Jupyter Kernels.")
else:
    import openmm
    print("\nenvironment OK -- openmm", openmm.__version__)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))

from saxsnmr import (Fetcher, prepare, run_md, benchmark, ensemble_fit,
                     frame_tables, radius_of_gyration, summarise_constructs,
                     load_trajectory, guinier, pair_distribution,
                     rg_from_pr, dmax_from_pr)
from saxsnmr.saxs import ScatteringTables, fit
from saxsnmr.nmr import (secondary_shifts, csi_assignment, best_offset,
                         threshold_scan)

# Simulation output is per system; the download cache is shared, so running a
# second notebook does not re-fetch what the first already has.
WORK = Path("work") / "rnase_a"
WORK.mkdir(parents=True, exist_ok=True)
fetcher = Fetcher(cache_dir=Path("work") / "cache")
UNIPROT = "P61823"

## 1. Which entries actually describe the same molecule?

A shared UniProt accession identifies a *protein*, not a *sample*. Entries
differ in construct, tags, truncation and oligomeric state.

Read this table before trusting any join. Look for disagreement in
`length`, and for `guinier_rg_A` / `pddf_dmax_A` that do not match the rest —
those entries are a different physical system. Note also that `angular_unit`
varies between entries; the loader normalises it, but never assume.

**On units.** SASBDB reports q in whichever unit `angular_unit` names, but
always reports Rg and Dmax in nanometres regardless. Mixing the two rescales
every length by ten. This package converts both to Angstrom on the way in and
marks the fields with an `_A` suffix, so the mistake cannot be made silently.

In [ ]:
codes = fetcher.search_sasbdb("ribonuclease")
print(f"{len(codes)} candidate entries")

survey = summarise_constructs(fetcher, codes[:12])
survey[["code", "uniprot", "length", "guinier_rg_A", "pddf_dmax_A",
        "angular_unit", "type_of_curve"]]

### Reading the survey

Monomeric RNase A is 124 residues with Rg ~15 A and Dmax ~46 A. The entries here were collected by SEC-SAXS, where the sample is separated immediately before measurement, so aggregates are far less of a concern than in a batch experiment -- the four entries agree on Rg to within 0.2 A.

In [ ]:
CODE = "SASDPF7"

curve = fetcher.sasbdb_curve(CODE)
print(curve)
print(f"SASBDB Guinier Rg = {curve.guinier_rg_A:.2f} +/- {curve.guinier_rg_err_A or float('nan'):.2f} A")
print(f"SASBDB P(r):  Rg = {curve.pddf_rg_A:.2f} A, Dmax = {curve.pddf_dmax_A:.1f} A")

# Restrict to where an implicit hydration shell is valid (see README).
curve = curve.trim(q_max=0.50)
print("after trim:", curve)

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(curve.q, curve.intensity, yerr=curve.sigma, fmt=".", ms=3,
            lw=0.5, alpha=0.7)
ax.set_yscale("log")
ax.set_xlabel("q  (1/A)")
ax.set_ylabel("I(q)  (arb.)")
ax.set_title(f"{CODE} experimental")
plt.tight_layout()

## 1b. Is this curve what it claims to be?

Before using an experimental curve to judge a model, check the curve. Two
standard diagnostics do most of the work, and neither depends on having a
structure.

**Guinier.** At low q, `ln I(q) = ln I0 - Rg^2 q^2 / 3`. The approximation
holds only while `q*Rg` is small — conventionally below about 1.3 for a
globular particle. That criterion is circular, since the valid range depends
on the Rg being measured, so `guinier()` extends the window point by point and
stops when the fitted Rg puts the last point past the limit. A curve with no
valid window is flagged rather than fitted: aggregation shows up here as an
upturn at the lowest q, and it would otherwise be reported as a large Rg with
no indication anything is wrong.

This recomputes Rg from the raw data rather than trusting the deposited value.
Agreement between the two is a check on the whole loading path — units
included.

In [ ]:
g = guinier(curve.q, curve.intensity, curve.sigma)
print(g)
print(f"  fitted {g.n_points} points, q = {g.q[0]:.4f} - {g.q[-1]:.4f} 1/A")
print(f"  SASBDB reports {curve.guinier_rg_A:.2f} A")
print(f"  difference: {abs(g.rg - curve.guinier_rg_A):.2f} A")
if g.note:
    print("  note:", g.note)

Note the error bars. This fit's formal error is far smaller than the one
SASBDB quotes, because it is the covariance of a straight line that assumes
the deposited sigmas are exactly right. The real uncertainty is dominated by
the choice of fitting range, which no covariance captures. `reduced_chi2` is
the more useful number: near 1 means the data really is a straight line over
the window; much above 1 means curvature, and therefore that the window is too
wide or the sample is not monodisperse.

In [ ]:
fig, (axg, axk) = plt.subplots(1, 2, figsize=(11, 4))

# Guinier: a straight line here, curvature means trouble.
axg.errorbar(g.q ** 2, g.ln_intensity, fmt=".", ms=4, label="experiment")
axg.plot(g.q ** 2, g.ln_model, lw=1.5, color="crimson",
         label=f"fit: Rg = {g.rg:.2f} A")
axg.set_xlabel("q^2  (1/A^2)")
axg.set_ylabel("ln I(q)")
axg.set_title(f"Guinier region (qRg {g.q_rg_min:.2f}-{g.q_rg_max:.2f})")
axg.legend()

# Kratky: a folded globular particle gives a bell with a clear maximum that
# returns towards the axis. A curve that rises and keeps rising indicates a
# disordered or unfolded chain -- worth knowing before blaming the force field.
axk.plot(curve.q, curve.q ** 2 * curve.intensity, lw=1)
axk.axvline(1.73 / g.rg, color="grey", ls=":",
            label="qRg = 1.73 (expected peak)")
axk.set_xlabel("q  (1/A)")
axk.set_ylabel("q^2 I(q)")
axk.set_title("Kratky -- folded or not?")
axk.legend(fontsize=8)
plt.tight_layout()

For a compact, well-folded particle the Kratky peak sits near `qRg = 1.73`.
If a target does not show that, the premise that one short trajectory
represents its solution ensemble is already in doubt, and a poor chi-squared
later would be a symptom rather than a cause.

## 2. Coordinates

SASBDB links a structure for only about 15% of entries, so coordinates come
from PDBe/SIFTS keyed on UniProt, falling back to AlphaFold.

Preparation strips pre-existing hydrogens, resolves alternate conformations and
selects a single chain. RNase A's best structure, `6etk` at 0.85 A, is comparable to lysozyme's reference: high enough resolution to carry its own hydrogens and alternate conformations. It has four disulfides, which the force field handles but which make the fold stiff.

The chain identifier is taken from the SIFTS mapping rather than assumed to be
`A`. For a protein solved inside a larger assembly it very often is not.

In [ ]:
hits = fetcher.best_structures(UNIPROT)[:5]
for h in hits:
    print(f"  {h.pdb_id} chain {h.chain_id}  {h.resolution} A  {h.method}")

CHAIN = hits[0].chain_id
structure, provenance = fetcher.structure_for(UNIPROT)
print("\nusing:", provenance, "chain", CHAIN)

prep = prepare(structure, WORK / "prepared.pdb", chain=CHAIN)
print(f"{prep.n_residues} residues, {prep.n_atoms} atoms")
print(prep.sequence)

## 3. How fast can this machine actually simulate?

Run this once. Throughput varies by more than an order of magnitude between
platforms, and every decision about trajectory length depends on the real number
rather than an estimate.

**Do not use OpenCL on integrated Intel graphics.** `python -m
openmm.testInstallation` on the development machine reported forces differing
from the CPU platform by 0.13, where CPU and Reference agree to 6e-06. A
platform that computes the wrong forces will still happily produce a
trajectory. Check your own output before trusting any platform other than CPU.

In [ ]:
rates = benchmark(prep, ps=2.0, platforms=["CPU"])   # OpenCL is wrong here; Reference is very slow
for name, ns_day in rates.items():
    print(f"  {name:<10} {ns_day:8.2f} ns/day" if np.isfinite(ns_day)
          else f"  {name:<10}      unavailable")

## 4. Short implicit-solvent MD

Implicit solvent is what puts this on a laptop. It also carries a known bias,
and Rg is exactly what SAXS measures most sensitively — so treat the fit below
as a check that the workflow is sound, not as a verdict on the force field.

**One setting worth knowing about.** `run_md` leaves Debye screening at
OpenMM's default, which is zero: the simulation has unscreened intramolecular
electrostatics regardless of the salt the experiment was measured in. Testing
this on lysozyme showed it does *not* explain the expansion seen there — but
it is a mismatch with the experiment either way, and the buffer conditions are
in the SASBDB entry if you want to check them for this system.

In [ ]:
# run_md's tuned defaults give roughly 4-8 ns/day for a system this size on a
# laptop CPU -- several times what the obvious settings give, mostly from a
# shorter nonbonded cutoff. Raise production_ps once the benchmark above tells
# you your rate.
result = run_md(prep, WORK, production_ps=5.0, equilibration_ps=2.0,
                frame_interval_ps=0.5, platform="CPU")
print(result)

In [ ]:
import mdtraj as mdt

# load_trajectory pairs the DCD with its topology and converts units.
traj = load_trajectory(result.trajectory, top=result.topology)
rg = radius_of_gyration(traj)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(len(rg)) * 0.5, rg, lw=1)
ax.axhline(g.rg, color="crimson", ls="--",
           label=f"experimental Guinier Rg = {g.rg:.2f} A")
ax.legend()
ax.set_xlabel("time (ps)")
ax.set_ylabel("Rg (A)")
ax.set_title("Compaction check")
plt.tight_layout()
print(f"simulated Rg = {rg.mean():.2f} +/- {rg.std():.2f} A")

## 5. SAXS: ensemble profile and fit

The experiment measures one ensemble-averaged curve, so the model must be
averaged over frames *before* fitting rather than fitting frames individually.

`c1` (excluded volume) and `c2` (hydration contrast) are free parameters. They
are reported alongside chi-squared deliberately — with two parameters floating,
a good fit alone does not establish much.

In [ ]:
ens = ensemble_fit(result, curve, stride=1)
print(ens)

static = load_trajectory(prep.path)
st = frame_tables(static, 0)
single = fit(st, curve.q, curve.intensity, curve.sigma)
print("single crystal structure:", single)

In [ ]:
fig, (ax, axr) = plt.subplots(2, 1, figsize=(6.5, 6), sharex=True,
                              gridspec_kw={"height_ratios": [3, 1]})
ax.errorbar(curve.q, curve.intensity, yerr=ens.sigma, fmt=".", ms=3, lw=0.5,
            alpha=0.5, label="experiment")
ax.plot(curve.q, single.model, lw=1.2, label=f"crystal structure, chi2={single.chi2:.2f}")
ax.plot(curve.q, ens.model, lw=1.5, label=f"MD ensemble, chi2={ens.chi2:.2f}")
ax.set_yscale("log")
ax.set_ylabel("I(q)")
ax.legend()
ax.set_title(f"{CODE} vs model")

axr.axhline(0, color="k", lw=0.5)
axr.plot(curve.q, (ens.model - curve.intensity) / ens.sigma, lw=1)
axr.set_xlabel("q  (1/A)")
axr.set_ylabel("residual / sigma")
plt.tight_layout()

## 5b. The same comparison in real space

Chi-squared is computed in reciprocal space, where a scale factor and two free
parameters are floating. P(r) — the distribution of interatomic distances —
gives an independent view with none of those adjustments. A model can be made
to fit I(q) reasonably while having the wrong shape; it is harder to hide that
in P(r).

Two caveats keep this honest. The experimental P(r) is not data: it is GNOM's
indirect Fourier transform of the intensity, and its shape depends on the Dmax
and regularisation the depositor chose. And the model's P(r) is weighted by
electron count with no hydration shell, where GNOM's reflects contrast against
solvent. Compare the *shape and extent*, not the fine detail.

In [ ]:
exp_pr = fetcher.pddf(CODE)
print("experiment:", exp_pr)

# Every fit carries the geometry it was computed from, so the ensemble-averaged
# tables are available here without rebuilding them.
r_x, p_x = pair_distribution(single.tables)
r_m, p_m = pair_distribution(ens.tables)

print(f"\n{'':22s} {'Rg (A)':>8s}  {'Dmax (A)':>9s}")
print(f"{'experiment (GNOM)':22s} {exp_pr.rg:8.2f}  {exp_pr.dmax:9.1f}")
print(f"{'crystal structure':22s} {rg_from_pr(r_x, p_x):8.2f}  {dmax_from_pr(r_x, p_x):9.1f}")
print(f"{'MD ensemble':22s} {rg_from_pr(r_m, p_m):8.2f}  {dmax_from_pr(r_m, p_m):9.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(exp_pr.r, exp_pr.normalised(), lw=2, color="k", label="experiment (GNOM)")
ax.fill_between(exp_pr.r,
                exp_pr.normalised() - exp_pr.error / exp_pr.p.max(),
                exp_pr.normalised() + exp_pr.error / exp_pr.p.max(),
                color="k", alpha=0.15, lw=0)
ax.plot(r_x, p_x, lw=1.3, label="crystal structure")
ax.plot(r_m, p_m, lw=1.3, label="MD ensemble")
ax.axvline(exp_pr.dmax, color="grey", ls=":", lw=1,
           label=f"experimental Dmax = {exp_pr.dmax:.0f} A")
ax.set_xlim(0, exp_pr.dmax * 1.25)
ax.set_xlabel("r  (A)")
ax.set_ylabel("P(r), normalised")
ax.set_title("Distance distribution")
ax.legend(fontsize=8)
plt.tight_layout()

A model whose P(r) extends past the experimental Dmax is too large — and
because SAXS is most sensitive to exactly that, it is the first thing to check
when chi-squared is poor.

## 5c. Neutron scattering — a prediction, not a validation

**There is no deposited SANS curve for RNase A.** SASBDB has none for this
molecule, so everything below is a forward prediction with nothing to check it
against. It is included because the contrast behaviour is informative in its
own right, and because the same code *is* validated against experiment in
`05_sans_validation.ipynb` — read that one before trusting these numbers.

Neutrons scatter from nuclei, so the scattering length carries no q
dependence — the calculation is simpler than the X-ray case, and reuses the
same distance histograms. What makes it physically different is hydrogen:

- **Hydrogen scatters negatively** (b = −3.74 fm) where deuterium is strongly
  positive (+6.67 fm). Nothing in X-ray scattering behaves like this.
- **Labile hydrogens exchange with solvent.** Those on N, O and S swap with the
  buffer and become deuterium in D₂O; carbon-bound ones do not. The two
  classes are therefore tracked separately, from the bond graph rather than
  from atom names.
- **Hydrogen scatters incoherently** — 80.3 barn against 1.76 coherent — which
  contributes a large flat background carrying no structural information.

In [ ]:
from saxsnmr import (sans_tables, sans_profile, sans_fit, sans_elements,
                    solvent_sld, match_point_d2o)
from saxsnmr.sans import SLD_H2O, SLD_D2O

labels = sans_elements(static.topology)
n_h = labels.count("H")
n_hx = labels.count("Hx")
print(f"hydrogens: {n_h} carbon-bound (fixed), {n_hx} on N/O/S (exchangeable)")
print(f"solvent SLD: H2O {SLD_H2O:+.4f}, D2O {SLD_D2O:+.4f} fm/A^3")

sans_tab = sans_tables(static, 0)
match = match_point_d2o(sans_tab)
print(f"\ncontrast match point for this molecule: {match:.1%} D2O")
print("Literature quotes 40-45% for a typical protein. This runs slightly")
print("higher because complete exchange of labile positions is assumed, which")
print("overstates deuteration -- buried amides exchange slowly or not at all.")

At the match point the molecule's mean scattering length density equals the
solvent's, the coherent signal very nearly vanishes, and what remains is
dominated by the hydration shell — which this implementation does not model.
**That is where these numbers are least trustworthy**, not merely noisiest.

In [ ]:
q_n = np.linspace(0.01, 0.30, 80)
fig, (axc, axi) = plt.subplots(1, 2, figsize=(11, 4))

for x in (0.0, 0.25, 0.42, 0.70, 1.0):
    axc.plot(q_n, sans_profile(sans_tab, q_n, x), lw=1.2, label=f"{x:.0%} D2O")
axc.axhline(0, color="k", lw=0.5)
axc.set_yscale("symlog")
axc.set_xlabel("q  (1/A)")
axc.set_ylabel("I(q), arbitrary units")
axc.set_title("Predicted contrast series")
axc.legend(fontsize=8)

frac = np.linspace(0, 1, 60)
i0 = [sans_profile(sans_tab, np.array([0.01]), x)[0] for x in frac]
axi.plot(frac, np.sqrt(np.abs(i0)), lw=1.5)
axi.axvline(match, color="crimson", ls="--", label=f"match point {match:.0%}")
axi.set_xlabel("D2O fraction")
axi.set_ylabel("sqrt(|I(0)|)  ~ |contrast|")
axi.set_title("Contrast vanishes at the match point")
axi.legend(fontsize=8)
plt.tight_layout()

The right panel is the signature of a contrast-variation experiment: the
square root of I(0) falls linearly to zero at the match point and rises again
beyond it. Reproducing that V shape is the weakest possible check on the
calculator — it only confirms the contrast arithmetic, not the shape of the
molecule.

## 6. NMR: chemical shifts against simulated secondary structure

Predicting raw shifts needs SPARTA+, SHIFTX2 or UCBShift, none of which builds
on Windows. The portable route is secondary chemical shifts: subtracting
random-coil values leaves a signal whose sign reports helix versus strand, which
can be compared with DSSP over the trajectory.

**This section needs controls more than any other.** An agreement percentage
on its own is uninterpretable: secondary structure classes are nowhere near
equally populated, so a model that always answers "helix" can score well while
containing no information at all. Two reference points are computed alongside
the simulation — the crystal structure, which is as close to correct as a
model gets, and a majority-class predictor, which is as close to useless as a
model gets. The MD result only means something between them.

### Choosing a BMRB entry

Several entries usually exist for one protein, and they are not
interchangeable. Taking the first with enough CA shifts is not good enough:
for RNase A that picks an entry whose secondary shifts have a standard
deviation of 0.59 ppm where five other entries give 3.6–5.0 ppm. An entry with
no dispersion is not reporting conformation — it may be a denatured or
dynamically averaged sample, or shifts that already had a reference subtracted
before deposition. Used unknowingly it produced an MCC of 0.009, and only the
controls revealed that.

Entries are therefore screened on `shift_dispersion`, which looks **only at the
NMR data**. Screening them on agreement with the structure would be fitting to
the answer.

In [ ]:
from saxsnmr import shift_dispersion

ids = fetcher.bmrb_ids(UNIPROT)
print(f"{len(ids)} BMRB entries for {UNIPROT}")

MIN_DISPERSION = 2.0        # ppm; see shift_dispersion's docstring

shifts, used, candidates = None, None, []
for bid in ids:
    df = fetcher.bmrb_shifts(bid)
    if df is None:
        continue
    atoms = df["atom"].str.upper()
    n_ca, n_cb = int((atoms == "CA").sum()), int((atoms == "CB").sum())
    if n_ca < 50 or n_cb < 50:
        continue
    sd = shift_dispersion(secondary_shifts(df))
    candidates.append((bid, n_ca, n_cb, sd))

# Take the most dispersed usable entry, not merely the first. Both criteria
# look only at the NMR data, so neither biases the comparison that follows.
usable = sorted([c for c in candidates if c[3] >= MIN_DISPERSION],
                key=lambda c: -c[3])

print(f"\n{'BMRB':>8s} {'CA':>5s} {'CB':>5s} {'sd (ppm)':>9s}   note")
for bid, n_ca, n_cb, sd in sorted(candidates, key=lambda c: -c[3]):
    note = ("chosen -- most dispersed" if usable and bid == usable[0][0]
            else "usable" if sd >= MIN_DISPERSION
            else "rejected -- no conformational dispersion")
    print(f"{bid:>8s} {n_ca:5d} {n_cb:5d} {sd:9.2f}   {note}")

if not usable:
    raise RuntimeError(
        "no BMRB entry has both CA and CB coverage and enough dispersion to "
        "be informative; the NMR comparison cannot be made for this system")
used = usable[0][0]
shifts = fetcher.bmrb_shifts(used)
print(f"\nusing BMRB {used}")
shifts.head()

In [ ]:
sec = secondary_shifts(shifts)
csi = csi_assignment(sec)

dssp = mdt.compute_dssp(traj, simplified=True)
resids = np.array([r.resSeq for r in traj.topology.residues])
offset, comparison = best_offset(csi, dssp, resids)
print(f"best numbering offset: {offset:+d}")

# The control: the same comparison against the crystal structure. A
# high-resolution experimental structure is about as right as a model gets, so
# it sets the ceiling this method can reach. If MD matches it, MD is not the
# limitation.
crystal_dssp = mdt.compute_dssp(static, simplified=True)
crystal_resids = np.array([r.resSeq for r in static.topology.residues])
_, crystal_cmp = best_offset(csi, crystal_dssp, crystal_resids)

print()
print(f"{'model':<20s} {'MCC':>7s} {'agree':>7s} {'baseline':>9s} {'balanced':>9s}")
for name, c in [("crystal structure", crystal_cmp), ("MD ensemble", comparison)]:
    print(f"{name:<20s} {c.mcc:7.3f} {c.agreement:7.1%} "
          f"{c.baseline_agreement:9.1%} {c.balanced_accuracy:9.1%}")

print(f"\nCSI assigns: {csi['csi'].value_counts().to_dict()}")
print(f"DSSP on the crystal: "
      f"{dict(zip(*np.unique(crystal_dssp[0], return_counts=True)))}")

Read **MCC** first — Matthews correlation, where 0 is chance and 1 is perfect.
It accounts for the class imbalance that makes raw agreement misleading. The
`baseline` column is what a predictor that always answers the commonest class
would score; an agreement near that number means nothing.

The confusion matrix shows *how* the two disagree, which a single percentage
hides.

In [ ]:
print("crystal structure — rows: NMR says, columns: structure says")
print(crystal_cmp.confusion)
print()
print("MD ensemble")
print(comparison.confusion)

### Does the answer depend on where the threshold is drawn?

The CSI cut-off is a judgement call, so the sensitivity is reported rather
than a single hand-picked number. If a conclusion only survives at one
threshold, it belongs to that choice and not to the structure.

The default of 1.75 ppm comes from the canonical secondary shifts: helix sits
near +3.5 ppm on this index and strand near −3.7, so the boundary is placed at
half the canonical magnitude. It is deliberately *not* tuned to maximise
agreement, which would be fitting to the answer.

In [ ]:
scan = threshold_scan(sec, crystal_dssp, crystal_resids)
print("against the crystal structure:")
print(scan.to_string(index=False, formatters={
    "MCC": "{:.3f}".format, "agreement": "{:.1%}".format,
    "baseline": "{:.1%}".format, "balanced": "{:.1%}".format}))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(scan["threshold"], scan["MCC"], "o-", label="MCC (crystal structure)")
ax.axvline(1.75, color="grey", ls=":", label="default threshold")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("CSI threshold on dCA - dCB  (ppm)")
ax.set_ylabel("Matthews correlation")
ax.set_ylim(min(0, scan["MCC"].min() * 1.2), max(0.6, scan["MCC"].max() * 1.2))
ax.legend(fontsize=8)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
t = comparison.table
colors = {"H": "crimson", "E": "steelblue", "C": "lightgrey"}
ax.bar(t["resid"], t["d_diff"], color=[colors.get(c, "grey") for c in t["csi"]])
ax.axhline(1.75, color="k", lw=0.5, ls=":")
ax.axhline(-1.75, color="k", lw=0.5, ls=":")
ax.set_xlabel("residue")
ax.set_ylabel("secondary shift  dCA - dCB (ppm)")
ax.set_title(f"NMR secondary shifts vs MD DSSP  "
             f"(MCC {comparison.mcc:.2f}, {comparison.agreement:.0%} agreement "
             f"vs {comparison.baseline_agreement:.0%} baseline, "
             f"n={comparison.n_compared})")
plt.tight_layout()

### Reading the NMR result honestly

Compare the simulation against the crystal structure, not against 100%. If the
crystal scores no better than the simulation, the comparison is bounded by the
method rather than by the model, and a poor number says nothing about the
dynamics. Strand recall in particular tends to be weak for any correct
structure: extended and polyproline-II backbone, which DSSP calls coil, carries
strand-like chemical shifts, so the index over-calls strand.

## 7. Mutations — and whether this method can see one

`prepare` takes mutations in PDBFixer's notation, so the same pipeline runs on
a variant. The interesting question is not whether it runs, but whether the
result means anything.

His12 and His119 are RNase A's catalytic pair, acting as general base and general acid. **H12A** abolishes activity. Like E35Q in lysozyme it is a chemical change rather than a shape change, so SAXS should not see it -- but unlike E35Q it removes a side chain rather than swapping one, so PDBFixer's rebuilt rotamer matters even less.

So the run below uses **replicates**, not a single pair. 3 wild-type
and 3 mutant simulations differ only in random seed, which estimates
the scatter instead of assuming it.

This matters more than it sounds. An earlier version of this notebook compared
one mutant run against two wild-type runs, and printed opposite conclusions on
consecutive executions of identical code — "within noise" one time, "worth
investigating" the next — because χ² at these trajectory lengths moves by
several units from seed alone. A control built from one pair has no power; it
just launders whichever answer the seeds happened to give.

3 per group is still a small sample. The cell below therefore reports the
smallest difference it could actually resolve, so a null result can be read as
"too noisy to tell" rather than "no effect".

In [ ]:
# PDBFixer rebuilds the side chain in a default rotamer without repacking its
# neighbours -- fine for a conservative substitution, poor for small-to-large
# ones. Check the sequence actually changed before trusting it.
mut = prepare(structure, WORK / "mutant.pdb", chain=CHAIN,
              mutations=["HIS-12-ALA"])
print(f"mutant: {mut.n_residues} residues, {mut.n_atoms} atoms")

wt_seq, mut_seq = prep.sequence, mut.sequence
diffs = [(i + 1, a, b) for i, (a, b) in enumerate(zip(wt_seq, mut_seq)) if a != b]
print("sequence differences (position, wild type, mutant):", diffs)
assert len(diffs) == 1, f"expected exactly one substitution, got {diffs}"

In [ ]:
# Same settings as the wild-type run above, so the comparison is like for like.
MD_KW = dict(production_ps=5.0, equilibration_ps=2.0,
             frame_interval_ps=0.5, platform="CPU")

# Replicates, not a single pair. Comparing one mutant run against one wild-type
# run cannot work: chi-squared varies by several units between runs that differ
# only in random seed, so a single pair produces whichever answer the seeds
# happen to give.
N_REPLICAS = 3
wt_runs, mut_runs = [], []
for i in range(N_REPLICAS):
    wt_runs.append(run_md(prep, WORK / f"wt_rep{i}", seed=i, **MD_KW))
    mut_runs.append(run_md(mut, WORK / f"mut_rep{i}", seed=100 + i, **MD_KW))
    print(f"replica {i}:  wt {wt_runs[-1].ns_per_day:5.2f} ns/day   "
          f"mutant {mut_runs[-1].ns_per_day:5.2f} ns/day", flush=True)

In [ ]:
import pandas as pd

def measure(res, label, replica):
    tr = load_trajectory(res.trajectory, top=res.topology)
    f_ = ensemble_fit(res, curve, stride=1)
    r_, p_ = pair_distribution(f_.tables)
    return {"group": label, "replica": replica,
            "Rg": radius_of_gyration(tr).mean(),
            "Dmax": dmax_from_pr(r_, p_),
            "chi2": f_.chi2, "c1": f_.c1, "c2": f_.c2}

rows = [measure(r, "wild type", i) for i, r in enumerate(wt_runs)]
rows += [measure(r, "H12A", i) for i, r in enumerate(mut_runs)]
per_run = pd.DataFrame(rows)
print(per_run.round(3).to_string(index=False))

summary = per_run.groupby("group")[["Rg", "Dmax", "chi2"]].agg(["mean", "std"])
print()
print(summary.round(3).to_string())

In [ ]:
wt = per_run[per_run["group"] == "wild type"]["chi2"]
mu = per_run[per_run["group"] == "H12A"]["chi2"]

# Pooled standard deviation, then the difference expressed in units of it.
# With this few replicas it is a crude estimate; the point is to compare the
# effect against the scatter rather than against zero.
pooled = np.sqrt(((len(wt) - 1) * wt.std(ddof=1) ** 2
                  + (len(mu) - 1) * mu.std(ddof=1) ** 2)
                 / (len(wt) + len(mu) - 2))
diff = mu.mean() - wt.mean()
effect = diff / pooled if pooled > 0 else np.nan

# Smallest difference these replicas could distinguish, roughly: with n per
# group, the standard error of the difference is pooled * sqrt(2/n).
sem_diff = pooled * np.sqrt(2.0 / len(wt))

print(f"wild type chi2 : {wt.mean():6.2f} +/- {wt.std(ddof=1):.2f}  (n={len(wt)})")
print(f"H12A      chi2 : {mu.mean():6.2f} +/- {mu.std(ddof=1):.2f}  (n={len(mu)})")
print(f"difference     : {diff:+6.2f}   ({effect:+.2f} pooled SD)")
print(f"\nstandard error of that difference : {sem_diff:.2f}")
print(f"smallest difference these replicas could resolve (~2 SE) : {2 * sem_diff:.2f}")

if abs(diff) < 2 * sem_diff:
    print("\nNot resolvable. The difference is smaller than this many replicas "
          "can distinguish from zero.")
else:
    print("\nThe difference exceeds what these replicas would produce by chance "
          "alone. That is a reason to run more replicas, not a result: with "
          f"n={len(wt)} per group the estimate of the spread is itself rough.")

### Reading this honestly

Three general rules this section illustrates, in increasing order of how often
they are forgotten:

1. **A difference smaller than the method's own run-to-run variation is not a
   measurement.** Always report the scatter next to the effect.
2. **A control needs replicates.** One run versus one run estimates nothing; it
   produces a number that looks like evidence. Watch what happens to the
   conclusion if `N_REPLICAS` is set to 1.
3. **A null result needs a detection limit.** "No significant difference" is
   only informative alongside the smallest difference the experiment could have
   found. Without it, a null is indistinguishable from an underpowered test.

Note also whether `c2` sits at its lower bound in every run. If it does, the
hydration term is doing no work and the comparison rests on excluded volume
and shape alone.

## 8. What this does and does not establish

**Supported by the run above**

- the fetch-and-join layer resolves a molecule to a curve, a structure and shifts
- the SAXS calculator reproduces an experimental curve from coordinates, and
  recovers the deposited Guinier Rg from the raw data independently
- the model's P(r) can be compared with GNOM's in both Rg and extent
- the pipeline runs on a variant, with replicates that show what size of
  difference would be meaningful

**Not supported**

- any claim about force field accuracy. Implicit solvent carries a known bias,
  and `c1`/`c2` absorb part of the remaining discrepancy.
- anything in the wide-angle regime, which is out of scope for this project.
- any statement about the mutant. There is no experimental curve for
  H12A to check against, and the predicted difference is within
  run-to-run noise.
- any claim that MD reproduces secondary structure *better or worse* than the
  crystal structure does, unless the two scored differently by more than the
  scatter.
- the formal error on the Guinier Rg as an uncertainty. It is the covariance of
  a straight line, roughly ten times smaller than the honest figure.
- The deposited curve is in 1/nm, not 1/A. The loader converts it from the `angular_unit` field, and the deposited P(r) is converted the same way -- GNOM writes r in whatever length unit matches the input q, which is a silent factor of ten if assumed.

**Comparing across systems**

The same analysis runs on every reference system in `notebooks/`. A conclusion
that holds for one molecule and not another is a property of that molecule; one
that holds for all of them is a property of the method. Neither is visible from
a single notebook, which is why there is more than one.